In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision.models as models
from torch.amp import autocast, GradScaler
from torch.utils.data import Subset
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

transform_test = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [ ]:
train_data = datasets.ImageFolder(
    root="/kaggle/input/datasets/jxyle1/genderdataset/GenderDataset/Train",
    transform=transform_train
)


test_data = datasets.ImageFolder(
    root="/kaggle/input/datasets/jxyle1/genderdataset/GenderDataset/Test",
    transform=transform_test
)

train_dataload = DataLoader(
    train_data,batch_size=64,shuffle=True,num_workers=2,persistent_workers=False, pin_memory=True
)
test_dataload = DataLoader(
    test_data,batch_size=64,shuffle=True,num_workers=2,persistent_workers=False,pin_memory=True
)

In [ ]:
class SqueezeE(nn.Module):
    def __init__(self,in_c):
        super().__init__()

        r = 16
        reduced_dim = max(1,in_c // r)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.se = nn.Sequential(
            nn.Conv2d(in_c,reduced_dim,1),
            nn.SiLU(),
            nn.Conv2d(reduced_dim,in_c,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        scale = self.pool(x)
        scale = self.se(scale)
        return x * scale

In [ ]:
class Block(nn.Module):
    def __init__(self,in_c,out_c,stride):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_c,out_c,kernel_size=3,stride=stride,padding=1),
            nn.BatchNorm2d(out_c),
            nn.SiLU(),

            nn.Conv2d(out_c,out_c,kernel_size=3,stride=1,padding=1),
            nn.BatchNorm2d(out_c),
            nn.SiLU()
    )
        self.skip = nn.Identity()

        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )
        else:
            self.skip = nn.Identity()
        self.act = nn.SiLU()
    
    def forward(self,x):
        return self.act(self.conv(x) + self.skip(x))
    

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = Block(3,32,stride=1)
        self.block2 = Block(32,64,stride=2)
        self.block3 = Block(64,128,stride=2)
        self.block4 = Block(128,256,stride=2)
        self.block5 = Block(256,512,stride=2)

        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fcl = nn.Linear(512,2)
        self.dropout = nn.Dropout(0.3)
        self.se1 = SqueezeE(128)
        self.se2 = SqueezeE(256)
        self.se3 = SqueezeE(512)

    def forward(self,x):
        #Blocks
        x = self.block1(x)
        x = self.block2(x)
        x = self.se1(self.block3(x))
        x = self.se2(self.block4(x))
        x = self.se3(self.block5(x))


        x = self.pool(x)
        x = torch.flatten(x,1)
        
        x = self.dropout(x)
        x = self.fcl(x)

        return x  


In [ ]:
from torch.amp import autocast, GradScaler
max_epochs = 15
net = CNN().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(net.parameters(),lr = 1e-4,weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max_epochs)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
scaler = GradScaler()
for epoch in range(max_epochs):
    print(f"Training Epoch {epoch}...")

    
    running_loss = 0.0

    for i,data in enumerate(train_dataload):
        inputs, labels = data
        inputs,labels = inputs.to(device),labels.to(device)
        optimizer.zero_grad()
        with autocast(device_type=device):
            outputs = net(inputs)
            loss = loss_function(outputs,labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
    epoch_loss = running_loss / len(train_dataload)
    scheduler.step()
    print(f"Loss: {running_loss/len(train_dataload):.2f}")

In [ ]:
val_loss = 0.0
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_dataload:
        images,labels = data
        images, labels = images.to(device), labels.to(device) 

        outputs = net(images)
        loss = loss_function(outputs,labels)

        val_loss += loss.item()

        _,predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy}%")

In [ ]:
import joblib

model_data = {
    "model_state_dict" : net.state_dict(),
    "class_to_idx": train_data.class_to_idx
}
import torch

torch.save(model_data, "/kaggle/working/genderecog_model.pth")